In [4]:
#Validate the network file and the edges file match, Prepare data files for zP analysis
import pandas as pd
import networkx as nx

DATA_DIR = "C:/Users/hnielsen/Desktop/Genomics"

df = pd.read_csv(f"{DATA_DIR}/SparCC/files/sparcc_fdr_significant_pairs_r_filt.csv")
pos = df[df['correlation'] > 0]

G = nx.Graph()
for _, row in pos.iterrows():
    G.add_edge(row['ASV_1'], row['ASV_2'], weight=row['correlation'])

communities = nx.algorithms.community.louvain_communities(
    G, weight='weight', resolution=1.0, seed=42
)
order = sorted(range(len(communities)), key=lambda i: -len(communities[i]))
remap = {old: new for new, old in enumerate(order)}
node_comm = {}
for old_i, comm in enumerate(communities):
    for node in comm:
        node_comm[node] = remap[old_i]

saved = pd.read_csv(f"{DATA_DIR}/Louvain/asv_community_assignments.csv")
check = saved.set_index('ASV')['community'].to_dict()

# Label-agnostic validation: instead of comparing raw ID numbers (which can
# differ between files even for an identical partition), check whether each
# fresh community maps cleanly onto exactly one saved community.
df_cmp = pd.DataFrame({'ASV': list(node_comm.keys()), 'fresh': list(node_comm.values())})
df_cmp['saved'] = df_cmp['ASV'].map(check)

ct = pd.crosstab(df_cmp['fresh'], df_cmp['saved'])
# For a perfect match (ignoring labels), every row and every column should
# have exactly one non-zero entry.
clean_match = (ct.astype(bool).sum(axis=1) == 1).all() and (ct.astype(bool).sum(axis=0) == 1).all()
print("Partition matches exactly (ignoring label numbering):", clean_match)
print(ct)

Partition matches exactly (ignoring label numbering): True
saved  0   1   2   3  4  5   6  7  8
fresh                               
0      0   0   0  50  0  0   0  0  0
1      0   0   0   0  0  0  42  0  0
2      0  31   0   0  0  0   0  0  0
3      0   0  20   0  0  0   0  0  0
4      0   0   0   0  0  9   0  0  0
5      0   0   0   0  5  0   0  0  0
6      4   0   0   0  0  0   0  0  0
7      0   0   0   0  0  0   0  2  0
8      0   0   0   0  0  0   0  0  2


In [6]:
# calculate zP scores
import pandas as pd
import numpy as np

DATA_DIR = "C:/Users/hnielsen/Desktop/Genomics"

# ---- Rebuild the positive-only network ----
df = pd.read_csv(f"{DATA_DIR}/SparCC/files/sparcc_fdr_significant_pairs_r_filt.csv")
pos = df[df['correlation'] > 0]

G = nx.Graph()
for _, row in pos.iterrows():
    G.add_edge(row['ASV_1'], row['ASV_2'], weight=row['correlation'])

# ---- Use the SAVED community labels directly (validated as an identical
# partition to a fresh Louvain run, just with different ID numbering) ----
saved = pd.read_csv(f"{DATA_DIR}/Louvain/asv_community_assignments.csv")
node_comm = saved.set_index('ASV')['community'].to_dict()

# sanity check: every node in the network has a saved community
missing = [n for n in G.nodes() if n not in node_comm]
print(f"Nodes in network with no saved community label: {len(missing)}")

modules = sorted(set(node_comm.values()))

# ---- Step 1: intra-module degree ----
k_i_module = {
    n: sum(1 for nb in G.neighbors(n) if node_comm.get(nb) == node_comm.get(n))
    for n in G.nodes()
}

# ---- Step 2: within-module degree z-score ----
z_score = {}
for s in modules:
    members = [n for n in G.nodes() if node_comm.get(n) == s]
    vals = np.array([k_i_module[n] for n in members])
    mean_s, std_s = vals.mean(), vals.std()
    for n in members:
        z_score[n] = (k_i_module[n] - mean_s) / std_s if std_s > 0 else 0.0

# ---- Step 3: participation coefficient ----
P_score = {}
for n in G.nodes():
    k_i = G.degree(n)
    if k_i == 0:
        P_score[n] = 0.0
        continue
    s_counts = {}
    for nb in G.neighbors(n):
        s_counts[node_comm.get(nb)] = s_counts.get(node_comm.get(nb), 0) + 1
    P_score[n] = 1 - sum((c / k_i) ** 2 for c in s_counts.values())

# ---- Step 4: classify roles ----
def categorize(z, P):
    if z >= 2.5:
        if P <= 0.30:
            return "Provincial hub"
        elif P <= 0.75:
            return "Connector hub"
        else:
            return "Kinless hub"
    else:
        if P <= 0.05:
            return "Ultra-peripheral non-hub"
        elif P <= 0.62:
            return "Peripheral non-hub"
        elif P <= 0.80:
            return "Non-hub connector"
        else:
            return "Non-hub kinless"

nodes_list = list(G.nodes())
result = pd.DataFrame({
    'ASV': nodes_list,
    'community': [node_comm.get(n) for n in nodes_list],
    'degree': [G.degree(n) for n in nodes_list],
    'intramodule_degree': [k_i_module[n] for n in nodes_list],
    'z_score': [z_score[n] for n in nodes_list],
    'P_score': [P_score[n] for n in nodes_list],
})
result['role'] = result.apply(lambda r: categorize(r['z_score'], r['P_score']), axis=1)

print(result['role'].value_counts())

result = result.sort_values(['role', 'z_score'], ascending=[True, False])
result.to_csv(f"{DATA_DIR}/zP/zP_analysis.csv", index=False)
print(f"\nSaved to {DATA_DIR}/zP/zP_analysis.csv")

Nodes in network with no saved community label: 0
role
Ultra-peripheral non-hub    101
Peripheral non-hub           64
Name: count, dtype: int64

Saved to C:/Users/hnielsen/Desktop/Genomics/zP/zP_analysis.csv
